# 장애인·노인 도보 400m E2SFCA 접근성

- 목적: 장애인친화시설과 노인 편의서비스 접근성을 별도 지표로 산출함.
- 순서: 장애인 접근성 → 노인 접근성.
- 네트워크: 도보 400m 이내 접근 pair만 사용함.

## 1. 사용 데이터와 수요 정의

- 장애인 수요: 문화누리대상자 중 장애인 추정 인구.
- 노인 수요: 문화누리대상자 중 70세 이상 추정 인구.
- 서울은 행정동 장애인비율을 사용함.
- 경기·인천은 시군구 등록장애인비율을 사용함.
- 최종 결과는 서울 격자 기준으로 저장함.

## 2. 시설 정의

- 장애인친화시설: `장애인친화시설 == 1`.
- 노인 편의서비스: `전화결제 == 1` 또는 `찾아가는문화서비스 == 1`.
- 같은 가맹점이 여러 중분류로 반복되면 1개 시설로 처리함.

## 3. E2SFCA 수식

$$
W(c_{ij})=
\begin{cases}
1.00 & 0 \le c_{ij} \le 150m \\
0.60 & 150m < c_{ij} \le 300m \\
0.25 & 300m < c_{ij} \le 400m \\
0 & c_{ij} > 400m
\end{cases}
$$

$$
R_j^g=\frac{S_j}{\sum_i D_i^gW(c_{ij})}
$$

$$
A_i^g=\sum_j R_j^gW(c_{ij})
$$

## 4. 실행

- 도보 400m pair를 불러옴.
- 장애인친화시설과 노인 편의서비스 시설을 분리함.
- 격자별 접근성 테이블과 시설별 공급수요비를 저장함.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.6f}".format)

BASE_PATH = Path().resolve()

if BASE_PATH.name == "access":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif (BASE_PATH / "analysis_table").exists():
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")

ANALYSIS_OUTPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "output"
NETWORK_OUTPUT_PATH = ANALYSIS_OUTPUT_PATH / "network_competition_25km"
ACCESS_OUTPUT_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT" / "disability_elderly_accessibility"
DOCS_PATH = PROJECT_PATH / "notebooks" / "access" / "docs"

ACCESS_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
DOCS_PATH.mkdir(parents=True, exist_ok=True)

GRID_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석격자.parquet"
STORE_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석가맹점.parquet"
WALK_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_도보접근성.parquet"
SUPPLY_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT" / "h3sfca" / "문화누리_가맹점_카테고리별_공급량.csv"
SEOUL_MNC_DISABLED_PATH = ANALYSIS_OUTPUT_PATH / "서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv"
EXTERNAL_AGE_PATH = ANALYSIS_OUTPUT_PATH / "인천경기_외부25km_100m_성연령별_추정인구.parquet"
DISABLED_SIGUNGU_PATH = PROJECT_PATH / "data" / "grid" / "시군구별_장애정도별_성별_등록장애인수_20260814052825.csv"
RESIDENT_POP_PATH = PROJECT_PATH / "data" / "grid" / "202410_202410_주민등록인구및세대현황_월간.csv"

WALK_CUTOFF_M = 400.0


def log(message):
    print(f"[장애인·노인 E2SFCA] {message}")


def clean_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )


def normalize_age(value):
    value = str(value)
    if value == "100세이상":
        return "100세-"
    return value


def walk_decay(cost):
    cost = pd.to_numeric(cost, errors="coerce")
    weight = np.zeros(len(cost), dtype="float32")
    weight[(cost >= 0) & (cost <= 150)] = 1.00
    weight[(cost > 150) & (cost <= 300)] = 0.60
    weight[(cost > 300) & (cost <= 400)] = 0.25
    return weight


def parse_sigungu_population():
    pop = pd.read_csv(RESIDENT_POP_PATH, encoding="cp949")
    pop["총인구수"] = clean_number(pop["2024년10월_총인구수"]).fillna(0)
    parsed = pop["행정구역"].astype(str).str.extract(r"^(?P<행정명>.+?)\s*\((?P<행정코드>\d+)\)$")
    pop["행정명"] = parsed["행정명"].str.strip()
    pop["행정코드"] = parsed["행정코드"]
    sigungu = pop[pop["행정코드"].str.endswith("00000", na=False)].copy()
    sigungu = sigungu[~sigungu["행정코드"].str.endswith("00000000", na=False)].copy()
    parts = sigungu["행정명"].str.split()
    sigungu["시도"] = parts.str[0]
    sigungu["시군구"] = parts.apply(lambda x: " ".join(x[1:]) if isinstance(x, list) and len(x) > 1 else np.nan)
    return sigungu[["시도", "시군구", "총인구수"]]


def parse_disabled_ratio():
    disabled = pd.read_csv(DISABLED_SIGUNGU_PATH, encoding="utf-8-sig")
    disabled = disabled.iloc[2:].copy()
    disabled = disabled.rename(columns={disabled.columns[0]: "시도", disabled.columns[1]: "시군구", disabled.columns[2]: "등록장애인수"})
    disabled["등록장애인수"] = clean_number(disabled["등록장애인수"]).fillna(0)
    disabled = disabled[disabled["시도"].isin(["서울특별시", "경기도", "인천광역시"])].copy()
    disabled = disabled[disabled["시군구"].ne("소계")].copy()
    pop = parse_sigungu_population()
    ratio = disabled[["시도", "시군구", "등록장애인수"]].merge(pop, on=["시도", "시군구"], how="left")
    ratio["장애인비율"] = np.where(ratio["총인구수"] > 0, ratio["등록장애인수"] / ratio["총인구수"], np.nan)
    sido_ratio = ratio.groupby("시도", as_index=False)["장애인비율"].mean().rename(columns={"장애인비율": "시도_장애인비율"})
    ratio = ratio.merge(sido_ratio, on="시도", how="left")
    ratio["장애인비율"] = ratio["장애인비율"].fillna(ratio["시도_장애인비율"]).fillna(0).clip(0, 1)
    return ratio[["시도", "시군구", "장애인비율"]]


def load_grid():
    grid = pd.read_parquet(GRID_COMPETITION_PATH).drop(columns=["geometry"], errors="ignore")
    grid["서울여부"] = grid["서울여부"].astype(bool)
    return grid


def load_store_supply():
    store = pd.read_parquet(STORE_COMPETITION_PATH).drop(columns=["geometry"], errors="ignore")
    for col in ["전화결제", "장애인친화시설", "찾아가는문화서비스", "노령인구_편의서비스"]:
        store[col] = pd.to_numeric(store[col], errors="coerce").fillna(0).astype(int)
    supply = pd.read_csv(SUPPLY_PATH, encoding="utf-8-sig")
    supply["공급량"] = pd.to_numeric(supply["공급량"], errors="coerce").fillna(1)
    store = store.merge(supply[["가맹점_ID", "중분류", "소분류", "공급량"]], on=["가맹점_ID", "중분류", "소분류"], how="left")
    ref = supply.groupby(["중분류", "소분류"], as_index=False)["공급량"].median().rename(columns={"공급량": "소분류_보정공급량"})
    cat = supply.groupby("중분류", as_index=False)["공급량"].median().rename(columns={"공급량": "중분류_보정공급량"})
    store = store.merge(ref, on=["중분류", "소분류"], how="left")
    store = store.merge(cat, on="중분류", how="left")
    store["공급량"] = store["공급량"].fillna(store["소분류_보정공급량"]).fillna(store["중분류_보정공급량"]).fillna(1).clip(lower=0)
    return store


def build_grid_demand(grid):
    seoul = pd.read_csv(
        SEOUL_MNC_DISABLED_PATH,
        usecols=[
            "GRID_CD", "시군구", "행정동", "연령대",
            "문화누리대상자_성연령별_추정_인구수",
            "문화누리대상자_성연령장애별_추정_인구수",
        ],
        encoding="utf-8-sig",
    )
    seoul["연령대"] = seoul["연령대"].map(normalize_age)
    elderly_ages = ["70-79세", "80-89세", "90-99세", "100세-"]
    seoul["노령인구_수요인구수"] = np.where(
        seoul["연령대"].isin(elderly_ages),
        pd.to_numeric(seoul["문화누리대상자_성연령별_추정_인구수"], errors="coerce").fillna(0),
        0,
    )
    seoul_demand = seoul.groupby(["GRID_CD", "시군구", "행정동"], as_index=False).agg(
        총_문화누리대상자=("문화누리대상자_성연령별_추정_인구수", "sum"),
        장애인_수요인구수=("문화누리대상자_성연령장애별_추정_인구수", "sum"),
        노령인구_수요인구수=("노령인구_수요인구수", "sum"),
    )

    disabled_ratio = parse_disabled_ratio()
    external = pd.read_parquet(EXTERNAL_AGE_PATH)
    external["연령대"] = external["연령대"].map(normalize_age)
    external = external.merge(disabled_ratio, on=["시도", "시군구"], how="left")
    sido_ratio = disabled_ratio.groupby("시도", as_index=False)["장애인비율"].mean().rename(columns={"장애인비율": "시도_장애인비율"})
    external = external.merge(sido_ratio, on="시도", how="left")
    external["장애인비율"] = external["장애인비율"].fillna(external["시도_장애인비율"]).fillna(0).clip(0, 1)
    external["총_문화누리대상자"] = pd.to_numeric(external["문화누리대상자_성연령별_추정_인구수"], errors="coerce").fillna(0)
    external["장애인_수요인구수"] = external["총_문화누리대상자"] * external["장애인비율"]
    external["노령인구_수요인구수"] = np.where(external["연령대"].isin(elderly_ages), external["총_문화누리대상자"], 0)
    external_demand = external.groupby(["GRID_CD", "시군구", "행정동"], as_index=False).agg(
        총_문화누리대상자=("총_문화누리대상자", "sum"),
        장애인_수요인구수=("장애인_수요인구수", "sum"),
        노령인구_수요인구수=("노령인구_수요인구수", "sum"),
    )
    demand = pd.concat([seoul_demand, external_demand], ignore_index=True)
    demand = demand.groupby(["GRID_CD", "시군구", "행정동"], as_index=False).sum()
    for col in ["총_문화누리대상자", "장애인_수요인구수", "노령인구_수요인구수"]:
        demand[col] = pd.to_numeric(demand[col], errors="coerce").fillna(0).clip(lower=0)
    log(f"수요 테이블: {demand.shape}")
    print(demand[["총_문화누리대상자", "장애인_수요인구수", "노령인구_수요인구수"]].sum())
    return demand


def load_walk_pair(store):
    pair = pd.read_parquet(WALK_PAIR_PATH)
    pair["접근비용"] = pd.to_numeric(pair["접근비용"], errors="coerce").astype("float32")
    pair = pair[pair["접근비용"].between(0, WALK_CUTOFF_M)].copy()
    facility_cols = [
        "가맹점_ID", "가맹점명", "중분류", "소분류",
        "전화결제", "장애인친화시설", "찾아가는문화서비스", "노령인구_편의서비스", "공급량"
    ]
    pair = pair.merge(store[facility_cols], on="가맹점_ID", how="left")
    pair["거리감쇠"] = walk_decay(pair["접근비용"])
    pair = pair[pair["거리감쇠"] > 0].copy()
    log(f"도보 400m 접근 pair: {pair.shape}, GRID {pair['GRID_CD'].nunique():,}, STORE {pair['가맹점_ID'].nunique():,}")
    return pair


def calculate_e2sfca(grid, pair, demand, indicator_name, flag_col, demand_col):
    target = pair[pair[flag_col].eq(1)].copy()
    if len(target) == 0:
        raise ValueError(f"{indicator_name} 대상 시설이 없습니다.")
    target = target.merge(demand[["GRID_CD", demand_col]], on="GRID_CD", how="left")
    target[demand_col] = target[demand_col].fillna(0)
    target["가중수요"] = target[demand_col] * target["거리감쇠"]
    # 동일 가맹점이 여러 중분류로 반복되면 편의서비스 지표에서는 1개 시설로 보되, 공급량은 가장 큰 보정공급량을 사용함.
    target = target.sort_values(["GRID_CD", "가맹점_ID", "공급량"], ascending=[True, True, False])
    target = target.drop_duplicates(["GRID_CD", "가맹점_ID"])
    facility = target.groupby("가맹점_ID", as_index=False).agg(
        가맹점명=("가맹점명", "first"),
        대표중분류=("중분류", "first"),
        공급량=("공급량", "max"),
        가중수요=("가중수요", "sum"),
    )
    facility["공급수요비"] = np.where(facility["가중수요"] > 0, facility["공급량"] / facility["가중수요"], 0)
    calc = target.merge(facility[["가맹점_ID", "공급수요비"]], on="가맹점_ID", how="left")
    calc["접근성기여"] = calc["공급수요비"].fillna(0) * calc["거리감쇠"]
    seoul_ids = set(grid.loc[grid["서울여부"], "GRID_CD"])
    access = calc[calc["GRID_CD"].isin(seoul_ids)].groupby("GRID_CD", as_index=False).agg(
        접근성지수=("접근성기여", "sum"),
        접근가능_가맹점수=("가맹점_ID", "nunique"),
        평균접근거리_m=("접근비용", "mean"),
    )
    access = access.rename(columns={
        "접근성지수": f"{indicator_name}_도보400m",
        "접근가능_가맹점수": f"{indicator_name}_도달가맹점수",
        "평균접근거리_m": f"{indicator_name}_평균접근거리_m",
    })
    facility["지표"] = indicator_name
    return access, facility


def save_docs():
    text = """# 장애인·노령인구 편의 접근성

## 사용 데이터
- 서울 + 외부 25km 100m 격자 수요 테이블
- 서울 문화누리대상자 성연령장애 추정 인구
- 외부 25km 문화누리대상자 성연령 추정 인구
- 경기·인천·서울 시군구 등록장애인 비율
- 서울 + 외부 25km 문화누리 가맹점
- 도보 네트워크 기준 400m 이내 격자-가맹점 pair

## 전처리/분석 방식
- 장애인 지표 수요는 문화누리대상자 중 장애인 추정 인구를 사용함.
- 노인 지표 수요는 문화누리대상자 중 70세 이상 추정 인구를 사용함.
- 장애인친화시설은 장애인친화시설 flag가 1인 가맹점을 사용함.
- 노인 편의서비스는 전화결제 또는 찾아가는문화서비스 중 하나라도 제공하는 가맹점을 사용함.
- 장애인·노인 모두 도보 400m 생활권만 적용함.
- 거리감쇠는 0~150m 1.00, 150~300m 0.60, 300~400m 0.25로 적용함.
- 같은 가맹점이 여러 중분류로 반복되면 편의서비스 접근성에서는 1개 시설로 처리함.

## 주요 산출물
- 장애인_노인_도보400m_E2SFCA_격자.csv
- 장애인_노인_도보400m_E2SFCA_가맹점_공급수요비.csv
- 장애인_노인_도보400m_E2SFCA_요약.csv
"""
    (DOCS_PATH / "disability_elderly_e2sfca_전처리_사용데이터.txt").write_text(text, encoding="utf-8")


def main():
    for path in [GRID_COMPETITION_PATH, STORE_COMPETITION_PATH, WALK_PAIR_PATH, SUPPLY_PATH, SEOUL_MNC_DISABLED_PATH, EXTERNAL_AGE_PATH, DISABLED_SIGUNGU_PATH, RESIDENT_POP_PATH]:
        if not path.exists():
            raise FileNotFoundError(path)
    grid = load_grid()
    store = load_store_supply()
    demand = build_grid_demand(grid)
    pair = load_walk_pair(store)
    disabled_access, disabled_facility = calculate_e2sfca(
        grid, pair, demand, "장애인친화시설", "장애인친화시설", "장애인_수요인구수"
    )
    elderly_access, elderly_facility = calculate_e2sfca(
        grid, pair, demand, "노인편의서비스", "노령인구_편의서비스", "노령인구_수요인구수"
    )
    seoul_base = grid[grid["서울여부"]][["GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y", "추정_인구수"]].copy()
    result = seoul_base.merge(demand[["GRID_CD", "총_문화누리대상자", "장애인_수요인구수", "노령인구_수요인구수"]], on="GRID_CD", how="left")
    result = result.merge(disabled_access, on="GRID_CD", how="left")
    result = result.merge(elderly_access, on="GRID_CD", how="left")
    fill_cols = [col for col in result.columns if col not in ["GRID_CD", "시군구", "행정동"]]
    result[fill_cols] = result[fill_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    facility = pd.concat([disabled_facility, elderly_facility], ignore_index=True)
    summary = result.agg({
        "총_문화누리대상자": ["sum", "mean", "max"],
        "장애인_수요인구수": ["sum", "mean", "max"],
        "노령인구_수요인구수": ["sum", "mean", "max"],
        "장애인친화시설_도보400m": ["mean", "median", "max"],
        "노인편의서비스_도보400m": ["mean", "median", "max"],
        "장애인친화시설_도달가맹점수": ["mean", "median", "max"],
        "노인편의서비스_도달가맹점수": ["mean", "median", "max"],
    })
    result.to_csv(ACCESS_OUTPUT_PATH / "장애인_노인_도보400m_E2SFCA_격자.csv", index=False, encoding="utf-8-sig")
    facility.to_csv(ACCESS_OUTPUT_PATH / "장애인_노인_도보400m_E2SFCA_가맹점_공급수요비.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(ACCESS_OUTPUT_PATH / "장애인_노인_도보400m_E2SFCA_요약.csv", encoding="utf-8-sig")
    save_docs()
    print("\n장애인·노인 접근성 결과 요약")
    display(summary)
    print("\n시설 수")
    print(facility.groupby("지표")["가맹점_ID"].nunique())
    print("\n상위 접근성 격자")
    display(result.sort_values("장애인친화시설_도보400m", ascending=False).head(10))
    return {"grid": result, "facility": facility, "summary": summary}


outputs = main()

## 5. 주요 결과 요약

- 서울 결과 격자: 60,528개.
- 장애인 수요인구 합: 27,431명.
- 노인 수요인구 합: 78,572명.
- 장애인친화시설 대상 가맹점: 114개.
- 노인 편의서비스 대상 가맹점: 310개.
- 장애인친화시설 도달 불가 격자: 58,832개.
- 노인 편의서비스 도달 불가 격자: 55,661개.
- 장애인친화시설 평균 접근성: 0.004568.
- 노인 편의서비스 평균 접근성: 0.006550.